# Getting data in and out

Every other tutorial starts from `mv.datasets`, which is convenient and not how
anyone's work actually begins. Real candidates come from a directory of CIFs, a
database query, an ASE trajectory, or a colleague's pickle.

This page is the full set of doors into and out of the object, and it queries a
real database rather than describing how one would. The point is that **matverse
is not a place your data gets stuck**: everything that goes in comes back out.

In [1]:
import matverse as mv
import numpy as np
import pandas as pd

mv.pl.set_style()

🔬 Starting plot initialization...
🧪 Calculators available: 6
    • emt — EMT (LGPL-2.1)
    • lj — Lennard-Jones (LGPL-2.1)
    • mace-mpa — mace-mpa (unstated)
    • mace-omat — mace-omat (unstated)
    • sevennet — sevennet (unstated)
    • chgnet — chgnet (unstated)
🖥️ NVIDIA CUDA GPUs: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3 — 79.1 GB, compute 9.0

                   __
   ____ ___  ____ _/ /__   _____  _____________
  / __ `__ \/ __ `/ __/ | / / _ \/ ___/ ___/ _ \
 / / / / / / /_/ / /_ | |/ /  __/ /  (__  )  __/
/_/ /_/ /_/\__,_/\__/ |___/\___/_/  /____/\___/

🔖 Version: 0.1.15   🧮 Functions: 144   📚 Tutorials: https://matverse.readthedocs.io/
✅ set_style complete.



## From a database, over OPTIMADE

[OPTIMADE](https://www.optimade.org/) is the common query API that most
materials databases now implement. Unlike the Materials Project's own REST API
it needs no key, so this is the door that works out of the box.

In [2]:
mv.data.optimade_providers()

{'mp': 'https://optimade.materialsproject.org/v1',
 'oqmd': 'https://oqmd.org/optimade/v1',
 'alexandria': 'https://alexandria.icams.rub.de/pbe/v1',
 'cod': 'https://www.crystallography.net/cod/optimade/v1',
 'mpds': 'https://api.mpds.io/v1',
 'nmd': 'https://nomad-lab.eu/prod/rae/optimade/v1',
 'jarvis': 'https://jarvis.nist.gov/optimade/jarvisdft/v1',
 'odbx': 'https://optimade.odbx.science/v1'}

We query [OQMD](https://oqmd.org/) — about 1.4 million entries — for every
binary aluminium–nickel structure it holds.

In [3]:
md = mv.data.from_optimade('elements HAS ALL "Al","Ni" AND nelements=2',
                           provider="oqmd", max_n=15)
md

AnnData object with n_obs × n_vars = 15 × 2
    obs: 'optimade_id', 'provider', 'formula', 'nelements'
    var: 'Z', 'atomic_mass', 'atomic_radius', 'electronegativity', 'group', 'period', 'melting_point', 'boiling_point', 'molar_volume', 'thermal_conductivity', 'electrical_resistivity', 'average_ionic_radius', 'max_oxidation_state', 'min_oxidation_state', 'is_metal', 'is_transition_metal', 'is_alkali', 'is_alkaline', 'is_metalloid', 'is_halogen', 'is_noble_gas', 'is_chalcogen', 'is_lanthanoid', 'is_actinoid', 'is_rare_earth_metal', 'block'
    uns: 'features', 'levels', 'provenance', 'X_is'
    obsm: 'structures'

In [4]:
mv.pp.describe(md)
md.obs[["optimade_id", "provider", "formula", "nsites", "density"]].round(3)

,optimade_id,provider,formula,nsites,density
0,4062279,oqmd,AlNi,2,5.914
1,4373031,oqmd,AlNi,2,5.920
2,4497420,oqmd,AlNi,2,5.692
3,4509489,oqmd,AlNi,2,5.918
4,4789722,oqmd,AlNi,2,5.910
5,4790088,oqmd,AlNi,2,5.924
6,5491454,oqmd,AlNi,2,5.044
7,5568716,oqmd,AlNi,4,5.927
8,5569350,oqmd,AlNi,2,3.765
9,5572689,oqmd,AlNi,4,4.208


Real data, with its real database identifiers in `obs['optimade_id']`, so any
row can be traced back to the record it came from.

```{note}
Provider endpoints go down, and matverse tries to tell you which kind of nothing
you got. Materials Project's OPTIMADE mirror currently answers `200` with
`data_returned=0` to *any* query including an empty filter — so "check your
filter" would be the wrong advice, and the error says so instead.
```

In [5]:
try:
    mv.data.from_optimade("nelements=2", provider="mp", max_n=5)
except ValueError as exc:
    print(f"ValueError: {exc}")

ValueError: the OPTIMADE endpoint returned no structures and reports data_returned=0, which means the provider itself is serving nothing rather than your filter matching nothing. Try another provider — mv.data.optimade_providers() lists them — or pass base_url= for an endpoint you trust.


In [6]:
try:
    mv.data.from_optimade("nelements=2", provider="nowhere")
except (KeyError, ValueError) as exc:
    print(f"{type(exc).__name__}: {exc}")

ValueError: unknown provider 'nowhere'; known: ['alexandria', 'cod', 'jarvis', 'mp', 'mpds', 'nmd', 'odbx', 'oqmd']. Any OPTIMADE endpoint works — pass base_url= directly.


### What a real query gives you that a curated one does not

Fifteen entries, all of them AlNi. Databases contain the same compound many
times — different calculations, different cells, different submissions — and
that is exactly what `mv.pp.dedup` is for.

In [7]:
mv.pp.qc(md)
mv.pp.dedup(md)
md.uns["dedup"]

{'source': 'input',
 'n_blocks': 7,
 'n_duplicates': 7,
 'n_unique': 8,
 'matcher': {'ltol': 0.2, 'stol': 0.3, 'angle_tol': 5.0}}

**Seven of fifteen were duplicates.** That is not a contrived example — it is
what happens when you query a real database and then pay a calculator to relax
everything you got.

`dedup` blocks on `(reduced formula, space group)` and runs pymatgen's
`StructureMatcher` inside each block, so the quadratic part stays local.

In [8]:
md.obs[["optimade_id", "formula", "nsites", "density", "is_duplicate"]].round(3)

,optimade_id,formula,nsites,density,is_duplicate
0,4062279,AlNi,2,5.914,False
1,4373031,AlNi,2,5.920,True
2,4497420,AlNi,2,5.692,False
3,4509489,AlNi,2,5.918,True
4,4789722,AlNi,2,5.910,True
5,4790088,AlNi,2,5.924,True
6,5491454,AlNi,2,5.044,False
7,5568716,AlNi,4,5.927,True
8,5569350,AlNi,2,3.765,False
9,5572689,AlNi,4,4.208,False


### Cached, so the second call is free

`mv.datasets.fetch` wraps the same query with a disk cache.

In [9]:
gold = mv.datasets.fetch('elements HAS ALL "Cu","Au" AND nelements=2',
                         provider="oqmd", max_n=8)
mv.pp.describe(gold)
gold.obs[["optimade_id", "formula", "nsites"]]

,optimade_id,formula,nsites
0,4061367,CuAu,2
1,4063155,CuAu,2
2,4475568,CuAu,2
3,4503006,CuAu,2
4,4514856,CuAu,2
5,4818972,CuAu,20
6,5493424,CuAu,2
7,5569482,CuAu,2


In [10]:
mv.datasets.cached()

[{'path': '/tmp/matverse/oqmd_elements_HAS_ALL__Cu___Au__AND_nelements_9849faa573.h5ad',
  'size_mb': 0.07}]

In [11]:
mv.datasets.cache_dir()

PosixPath('/tmp/matverse')

```{warning}
Set **`MATVERSE_DATA`** on a cluster. The default sits under the system
temporary directory, which on a compute node is usually node-local and wiped
when the job ends — so the cache silently never hits. `$SCRATCH` is where a
downloaded corpus belongs; a home directory, which is small, NFS-backed and
shared, is where it does not.
```

### Materials Project directly

`mv.data.from_mp` uses the native API, which returns computed properties
OPTIMADE does not carry — formation energies, band gaps, magnetic moments. It
needs `MP_API_KEY` in the environment.

```python
md = mv.data.from_mp({'elements': ['Li', 'Fe', 'P', 'O'], 'num_elements': 4})
```

### Parsing a payload you already have

If you fetched the JSON yourself — through a proxy, from a cache, from a
provider matverse does not know — `from_optimade_response` takes the parsed
dictionary. No network.

In [12]:
response = {"data": [
    {"id": "mp-30", "attributes": {
        "chemical_formula_reduced": "Cu",
        "lattice_vectors": [[0.0, 1.8075, 1.8075],
                            [1.8075, 0.0, 1.8075],
                            [1.8075, 1.8075, 0.0]],
        "cartesian_site_positions": [[0.0, 0.0, 0.0]],
        "species_at_sites": ["Cu"],
        "nsites": 1,
    }},
]}

parsed = mv.data.from_optimade_response(response)
mv.pp.describe(parsed)
parsed.obs[["formula", "nsites", "volume"]].round(3)

,formula,nsites,volume
0,Cu,1,11.81


## From structures you already have

In [13]:
from pymatgen.core import Lattice, Structure

structures = [
    Structure(Lattice.cubic(3.615), ["Cu"] * 4,
              [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
    Structure(Lattice.cubic(4.050), ["Al"] * 4,
              [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
]
local = mv.data.from_structures(structures, obs=pd.DataFrame({
    "sample": ["ICSD-627117", "ICSD-43423"],
    "measured_a": [3.6149, 4.0495],
}))
local.obs

,sample,measured_a
0,ICSD-627117,3.6149
1,ICSD-43423,4.0495


Pass a DataFrame and it becomes `obs`, aligned by position.

### From an iterable, when the list does not fit

`from_iterable` consumes a generator, so a million structures never all exist
at once.

In [14]:
def generated():
    """Stand-in for a reader that streams from disk."""
    for a in (3.5, 3.6, 3.7, 3.8):
        yield Structure(Lattice.cubic(a), ["Cu"] * 4,
                        [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]])


stream = mv.data.from_iterable(generated())
stream.n_obs, mv.provenance(stream)[-1]

(4, 'data.from_iterable(chunk_size=5000, n=4, n_blocks=1)')

## From ASE

ASE `Atoms` are the other structure object in the ecosystem, and every
calculator speaks them.

In [15]:
from ase.build import bulk

from_atoms = mv.data.from_ase([bulk("Cu", "fcc", a=3.615, cubic=True),
                               bulk("Ni", "fcc", a=3.524, cubic=True)])
mv.pp.describe(from_atoms)
from_atoms.obs[["formula", "nsites", "density"]].round(3)

,formula,nsites,density
0,Cu,4,8.935
1,Ni,4,8.908


### From a trajectory or structure file

`from_ase_file` reads anything ASE can — `.traj`, `.xyz`, `.cif`, VASP
`POSCAR` — so a molecular-dynamics trajectory becomes a dataset whose rows are
frames.

In [16]:
import tempfile
from pathlib import Path
from ase.io import write

workdir = Path(tempfile.mkdtemp())
write(workdir / "frames.xyz", [bulk("Cu", "fcc", a=a, cubic=True)
                               for a in (3.55, 3.60, 3.65, 3.70)])

frames = mv.data.from_ase_file(workdir / "frames.xyz", index=":")
mv.pp.describe(frames)
frames.obs[["formula", "volume"]].round(3)

,formula,volume
0,Cu,44.739
1,Cu,46.656
2,Cu,48.627
3,Cu,50.653


## CIFs, out and back

A directory of CIFs is how crystallographic data usually travels.

In [17]:
mv.data.to_cif(md, workdir / "cifs")
sorted(p.name for p in (workdir / "cifs").iterdir())[:6]

['0.cif', '1.cif', '10.cif', '11.cif', '12.cif', '13.cif']

In [18]:
round_trip = mv.data.from_cif(workdir / "cifs")
mv.pp.describe(round_trip)
round_trip.obs[["formula", "nsites", "density"]].head(5).round(3)

,formula,nsites,density
0,AlNi,2,5.914
1,AlNi,2,5.920
2,AlNi,2,5.769
3,AlNi,2,5.939
4,AlNi,2,5.920


Out and back with the formulas intact. `to_cif` names each file from its row,
which is what makes the round trip *identifiable* rather than merely possible.

## Out to the rest of the ecosystem

In [19]:
atoms = mv.data.to_ase(from_atoms)
atoms[0], atoms[0].get_chemical_symbols()

(MSONAtoms(symbols='Cu4', pbc=True, cell=[3.615, 3.615, 3.615]),
 ['Cu', 'Cu', 'Cu', 'Cu'])

In [20]:
back = mv.data.to_pymatgen(from_atoms)
back[0].composition.reduced_formula, round(back[0].lattice.a, 4)

('Cu', 3.615)

And the object itself is an ordinary `AnnData`, so `write_h5ad` is the archive
format — readable by anndata with matverse absent.

## matminer

[matminer](https://hackingmaterials.lbl.gov/matminer/) is the established
featurisation library, and matverse does not try to replace it: `to_matminer`
hands it a DataFrame, `from_matminer` takes one back, and `mv.feat.matminer`
runs its featurisers against this object.

In [21]:
try:
    frame = mv.data.to_matminer(from_atoms)
    print(frame.head())
except ImportError as exc:
    print(f"matminer is optional and absent here: {exc}")

                                           structure formula  nsites  \
0  [[0. 0. 0.] Cu, [0.     1.8075 1.8075] Cu, [1....      Cu       4   
1  [[0. 0. 0.] Ni, [0.    1.762 1.762] Ni, [1.762...      Ni       4   

      volume   density  n_elements  volume_per_atom  
0  47.241633  8.934544           1        11.810408  
1  43.763062  8.908214           1        10.940765  


In [22]:
try:
    rebuilt = mv.data.from_matminer(frame)
    print(rebuilt)
except (ImportError, NameError) as exc:
    print(f"needs matminer: {type(exc).__name__}")

AnnData object with n_obs × n_vars = 2 × 2
    obs: 'formula'
    var: 'Z', 'atomic_mass', 'atomic_radius', 'electronegativity', 'group', 'period', 'melting_point', 'boiling_point', 'molar_volume', 'thermal_conductivity', 'electrical_resistivity', 'average_ionic_radius', 'max_oxidation_state', 'min_oxidation_state', 'is_metal', 'is_transition_metal', 'is_alkali', 'is_alkaline', 'is_metalloid', 'is_halogen', 'is_noble_gas', 'is_chalcogen', 'is_lanthanoid', 'is_actinoid', 'is_rare_earth_metal', 'block'
    uns: 'features', 'levels', 'provenance', 'X_is'
    obsm: 'structures', 'X_matminer'


In [23]:
try:
    mv.feat.matminer(from_atoms, featurizers=["ElementProperty"])
    print(from_atoms.obsm["X_matminer"].shape)
except (ImportError, Exception) as exc:
    print(f"{type(exc).__name__}: {str(exc)[:90]}")

ImportError: mv.feat.matminer needs `pip install matverse[matminer]`


```{note}
matminer is an optional dependency (`pip install "matverse[matminer]"`), and an
absent optional backend produces a message naming what to install rather than a
traceback from three layers down.
```

## What survives a round trip

In [24]:
print(mv.utils.summary(md))

matverse dataset: 15 materials x 2 elements
  elements   Al, Ni
  structures input
  levels
    oqmd             OPTIMADE provider oqmd
  provenance 4 operations
    data.from_optimade
    pp.describe(source='input')
    pp.qc(source='input', min_distance=0.5, require_charge_balance=False)
    pp.dedup(source='input', symprec=0.1)


```{seealso}
[Getting started](getting_started.ipynb) is the pipeline these doors lead into;
[Infrastructure](infrastructure.ipynb) covers units, checkpoints and getting a
screen onto a cluster.
```